In [18]:
!pip install pyperplan

In [20]:
domain_pddl = """
(define (domain BLOCKS)
  (:requirements :strips :typing)
  (:types block)

  (:predicates
    (on ?x - block ?y - block)
    (ontable ?x - block)
    (clear ?x - block)
    (holding ?x - block)
    (handempty)
  )

  (:action pick-up
    :parameters (?x - block)
    :precondition
      (and
        (clear ?x)
        (ontable ?x)
        (handempty)
      )
    :effect
      (and
        (not (clear ?x))
        (not (ontable ?x))
        (not (handempty))
        (holding ?x)
      )
  )

  (:action put-down
    :parameters (?x - block)
    :precondition
      (holding ?x)
    :effect
      (and
        (not (holding ?x))
        (handempty)
        (ontable ?x)
        (clear ?x)
      )
  )

  (:action stack
    :parameters (?x - block ?y - block)
    :precondition
      (and
        (holding ?x)
        (clear ?y)
      )
    :effect
      (and
        (not (holding ?x))
        (handempty)
        (clear ?x)
        (not (clear ?y))
        (on ?x ?y)
      )
  )

  (:action unstack
    :parameters (?x - block ?y - block)
    :precondition
      (and
        (on ?x ?y)
        (clear ?x)
        (handempty)
      )
    :effect
      (and
        (holding ?x)
        (clear ?y)
        (not (clear ?x))
        (not (handempty))
        (not (on ?x ?y))
      )
  )
)
"""

with open("domain.pddl", "w") as f:
    f.write(domain_pddl)

In [21]:
problem_pddl = """
(define (problem BLOCK-6)
  (:domain BLOCKS)

  (:objects A B C D E F - block)

  (:init
    (handempty)
    (clear A)
    (clear D)
    (clear F)
    (ontable C)
    (ontable D)
    (ontable B)
    (on A E)
    (on E C)
    (on F B)
  )

  (:goal
    (and
      (on B C)
      (on C F)
      (on F A)
      (on D E)
    )
  )
)
"""

with open("problem.pddl", "w") as f:
    f.write(problem_pddl)

In [22]:
!pyperplan domain.pddl problem.pddl


2026-06-12 12:43:08,092 INFO     using search: breadth_first_search
2026-06-12 12:43:08,092 INFO     using heuristic: None
2026-06-12 12:43:08,092 INFO     Parsing Domain c:\Users\jinso.JINSUWON\Paper_Study_Note\PDDL\domain.pddl
2026-06-12 12:43:08,093 INFO     Parsing Problem c:\Users\jinso.JINSUWON\Paper_Study_Note\PDDL\problem.pddl
2026-06-12 12:43:08,094 INFO     5 Predicates parsed
2026-06-12 12:43:08,094 INFO     4 Actions parsed
2026-06-12 12:43:08,094 INFO     6 Objects parsed
2026-06-12 12:43:08,094 INFO     0 Constants parsed
2026-06-12 12:43:08,094 INFO     Grounding start: block-6
2026-06-12 12:43:08,095 INFO     Relevance analysis removed 0 facts
2026-06-12 12:43:08,095 INFO     Grounding end: block-6
2026-06-12 12:43:08,095 INFO     55 Variables created
2026-06-12 12:43:08,095 INFO     84 Operators created
2026-06-12 12:43:08,095 INFO     Search start: block-6
2026-06-12 12:43:08,157 INFO     Goal reached. Start extraction of solution.
2026-06-12 12:43:08,158 INFO     487

In [23]:
# !ls # 코랩에서 사용한 리눅스 명령어
from pathlib import Path

for file in Path(".").iterdir():
    print(file.name)

breadth_first_search_blockworld_steps
domain.pddl
PDDL.ipynb
problem.pddl
problem.pddl.soln


In [26]:
# !cat problem.pddl.soln # 코랩에서 사용한 리눅스 명령어
!type problem.pddl.soln

(unstack a e)
(put-down a)
(unstack f b)
(stack f a)
(unstack e c)
(put-down e)
(pick-up c)
(stack c f)
(pick-up b)
(stack b c)
(pick-up d)
(stack d e)


In [29]:
!pip install matplotlib

from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches

stacks = [["C", "E", "A"], ["D"], ["B", "F"]]

plan = [
    ("unstack", "a", "e"),
    ("put-down", "a"),
    ("unstack", "e", "c"),
    ("put-down", "e"),
    ("pick-up", "d"),
    ("stack", "d", "e"),
    ("unstack", "f", "b"),
    ("stack", "f", "a"),
    ("pick-up", "c"),
    ("stack", "c", "f"),
    ("pick-up", "b"),
    ("stack", "b", "c")
]

out_dir = Path("breadth_first_search_blockworld_steps")
out_dir.mkdir(exist_ok=True)

holding = None

def normalize(block):
    return block.upper()

def find_stack(block):
    block = normalize(block)
    for i, stack in enumerate(stacks):
        if block in stack:
            return i
    return None

def remove_empty_stacks():
    while [] in stacks:
        stacks.remove([])

def draw_state(step, action=None):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.set_xlim(0, max(5, len(stacks) + 2))
    ax.set_ylim(0, 7)
    ax.axis("off")

    block_w = 0.8
    block_h = 0.45

    ax.text(0.1, 6.5, f"Step {step}", fontsize=14)
    if action:
        ax.text(0.1, 6.1, "Action: " + " ".join(action), fontsize=11)

    for i, stack in enumerate(stacks):
        x = i + 1
        ax.plot([x - 0.5, x + 0.5], [0.5, 0.5])
        for j, block in enumerate(stack):
            y = 0.6 + j * block_h
            rect = patches.Rectangle((x - block_w/2, y), block_w, block_h, fill=False)
            ax.add_patch(rect)
            ax.text(x, y + block_h/2, block, ha="center", va="center", fontsize=12)

    if holding:
        ax.text(len(stacks) + 1, 5.5, f"Holding: {holding}", fontsize=12)

    plt.savefig(out_dir / f"breadth_step_{step:02d}.png", bbox_inches="tight")
    plt.close()

def apply_action(action):
    global holding

    op = action[0]
    args = [normalize(x) for x in action[1:]]

    if op == "unstack":
        x, y = args
        idx = find_stack(x)
        stacks[idx].pop()
        holding = x

    elif op == "put-down":
        x = args[0]
        stacks.append([x])
        holding = None
        remove_empty_stacks()

    elif op == "pick-up":
        x = args[0]
        idx = find_stack(x)
        stacks[idx].pop()
        holding = x
        remove_empty_stacks()

    elif op == "stack":
        x, y = args
        idx = find_stack(y)
        stacks[idx].append(x)
        holding = None

draw_state(0)

for_step = 1
for action in plan:
    apply_action(action)
    draw_state(for_step, action)
    for_step += 1

print(f"Saved images to: {out_dir}")

Saved images to: breadth_first_search_blockworld_steps
